<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Launch Sites Locations Analysis with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:
- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
import folium
import requests
import pandas as pd
import os


In [2]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/DV0101EN-3-5-1-Generating-Maps-in-Python-py-v2.0.ipynb)


## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site. 


In [3]:
# 1. Definición de parámetros (Configuración)
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
FILE_NAME = 'spacex_launch_geo.csv'

def download_spacex_data(url: str, output_path: str):
    """Descarga el dataset usando la librería requests con manejo de errores."""
    try:
        response = requests.get(url, timeout=10)
        # Verificamos si la descarga fue exitosa (Status 200)
        response.raise_for_status() 
        
        with open(output_path, 'wb') as f:
            f.write(response.content)
        print(f"[LOG] Archivo '{output_path}' descargado correctamente.")
        
    except requests.exceptions.RequestException as e:
        print(f"[ERROR] Fallo en la descarga: {e}")
        return None

# 2. Ejecución del flujo de Ingesta
if not os.path.exists(FILE_NAME):
    download_spacex_data(URL, FILE_NAME)
else:
    print(f"[LOG] El archivo '{FILE_NAME}' ya existe localmente. Omitiendo descarga.")

# 3. Lectura del DataFrame (Estructura de Datos)
spacex_df = pd.read_csv(FILE_NAME)

# Verificación rápida del esquema
print("\n--- Vista previa del dataset geoespacial ---")
print(spacex_df.head())

[LOG] El archivo 'spacex_launch_geo.csv' ya existe localmente. Omitiendo descarga.

--- Vista previa del dataset geoespacial ---
   Flight Number        Date Time (UTC) Booster Version  Launch Site  \
0              1  2010-06-04   18:45:00  F9 v1.0  B0003  CCAFS LC-40   
1              2  2010-12-08   15:43:00  F9 v1.0  B0004  CCAFS LC-40   
2              3  2012-05-22    7:44:00  F9 v1.0  B0005  CCAFS LC-40   
3              4  2012-10-08    0:35:00  F9 v1.0  B0006  CCAFS LC-40   
4              5  2013-03-01   15:10:00  F9 v1.0  B0007  CCAFS LC-40   

                                             Payload  Payload Mass (kg)  \
0               Dragon Spacecraft Qualification Unit                0.0   
1  Dragon demo flight C1, two CubeSats,  barrel o...                0.0   
2                             Dragon demo flight C2+              525.0   
3                                       SpaceX CRS-1              500.0   
4                                       SpaceX CRS-2           

Now, you can take a look at what are the coordinates for each site.


In [4]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [5]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example, 


In [6]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle. 


Now, let's add a circle for each launch site in data frame `launch_sites`


_TODO:_  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [7]:

# 1. Inicialización del mapa (NASA Coordinate como punto de referencia central)
# nasa_coordinate = [29.55968488, -95.08309718]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# 2. Iteración sobre el nuevo dataframe launch_sites_df
for index, row in launch_sites_df.iterrows():
    # Extraemos la coordenada del sitio actual
    coordinate = [row['Lat'], row['Long']]
    site_name = row['Launch Site']
    
    # A. Añadir el Círculo de Seguridad (Visualización de Radio de 1km)
    circle = folium.Circle(
        location=coordinate, 
        radius=1000, 
        color='#d35400', 
        fill=True
    ).add_child(folium.Popup(site_name))
    
    # B. Añadir el Marcador con el Nombre del Sitio (Estética Senior)
    marker = folium.map.Marker(
        location=coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 14pt; color:#d35400; width: 200px;"><b>%s</b></div>' % site_name,
        )
    )
    
    # C. Integrar elementos al objeto mapa
    site_map.add_child(circle)
    site_map.add_child(marker)

# 3. Renderizar el mapa
site_map

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:
- Are all launch sites in proximity to the Equator line?
- Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [11]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records. 
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [12]:
marker_cluster = MarkerCluster()


_TODO:_ Create a new column in `launch_sites` dataframe called `marker_color` to store the marker colors based on the `class` value


In [13]:

# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red
# Función para asignar colores según el éxito físico del lanzamiento
def assign_marker_color(launch_class):
    if launch_class == 1:
        return 'green'  # Éxito: Todo en orden
    else:
        return 'red'    # Fallo: Requiere análisis de residuos

# Aplicamos la función a spacex_df para crear la nueva columna
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)

# Verificamos la transformación (Rigor de analista)
print(spacex_df[['Launch Site', 'class', 'marker_color']].tail(10))


     Launch Site  class marker_color
46    KSC LC-39A      1        green
47    KSC LC-39A      1        green
48    KSC LC-39A      1        green
49  CCAFS SLC-40      1        green
50  CCAFS SLC-40      1        green
51  CCAFS SLC-40      0          red
52  CCAFS SLC-40      0          red
53  CCAFS SLC-40      0          red
54  CCAFS SLC-40      1        green
55  CCAFS SLC-40      0          red


In [ ]:
# Function to assign color to launch outcome
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

_TODO:_ For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [14]:
# 1. Aseguramos que el clúster esté añadido al mapa base
site_map.add_child(marker_cluster)

# 2. Iteración sobre spacex_df para poblar el clúster
# Usamos iterrows() para procesar cada registro de lanzamiento individual
for index, record in spacex_df.iterrows():
    # Definimos la coordenada [Latitud, Longitud]
    coordinate = [record['Lat'], record['Long']]
    
    # Creamos el objeto Marker con personalización dinámica
    # El color del icono vendrá de la columna 'marker_color' que creamos antes
    marker = folium.Marker(
        location=coordinate,
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        # Añadimos un popup con información técnica relevante
        popup=f"Site: {record['Launch Site']}<br>Outcome: {'Success' if record['class']==1 else 'Failure'}"
    )
    
    # 3. Añadimos el marcador al clúster (NO directamente al mapa)
    # Esto permite que Folium gestione el agrupamiento automático por cercanía
    marker_cluster.add_child(marker)

# 4. Renderizamos el mapa final
site_map

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [16]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


You can calculate the distance between two points on the map based on their `Lat` and `Long` values using the following method:


In [17]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

_TODO:_ Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [19]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
# 1. Definimos las coordenadas (Ejemplo: CCAFS LC-40 y su punto de costa más cercano)
launch_site_lat = 28.562302
launch_site_lon = -80.577356
coastline_lat = 28.56367
coastline_lon = -80.57163

# 2. Calculamos la distancia física usando tu función Haversine
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# 3. Crear un marcador en el punto de la costa para visualizar el objetivo
coast_coordinate = [coastline_lat, coastline_lon]
distance_marker = folium.Marker(
    location=coast_coordinate,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12pt; color:#283593;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
    )
)
site_map.add_child(distance_marker)

# 4. Dibujar una línea (PolyLine) entre el sitio de lanzamiento y la costa
# Esto visualiza el "vector" de distancia mínima
lines = folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]],
    weight=2,
    color='blue'
)
site_map.add_child(lines)

site_map

_TODO:_ After obtained its coordinate, create a `folium.Marker` to show the distance


In [20]:
# 1. Definimos las coordenadas del punto de costa que seleccionaste con MousePosition
# (Asegúrate de que estas variables coincidan con tu celda anterior)
coastline_lat = 28.56367
coastline_lon = -80.57163
launch_site_lat = 28.562302
launch_site_lon = -80.577356

# 2. Calculamos la distancia (llamando a tu función Haversine)
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# 3. Creamos el marcador de distancia (Distance Marker)
# Usamos un DivIcon con HTML para que el texto sea visible sin hacer clic
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(150, 36), # Tamaño suficiente para que el texto no se corte
        icon_anchor=(0, 0),
        html='<div style="font-size: 14pt; color:#d35400; font-weight: bold;">%s</div>' % "{:10.2f} KM".format(distance_coastline),
    )
)

# 4. Añadimos el marcador al mapa
site_map.add_child(distance_marker)

# 5. Opcional: Dibujamos la línea para conectar ambos puntos y visualizar el vector
lines = folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]], 
    weight=2,
    color='blue'
)
site_map.add_child(lines)

site_map

In [23]:
# Limpiamos el clúster anterior para evitar duplicados
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

# Iteramos sobre el dataframe detallado
for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color'])
    )
    marker_cluster.add_child(marker)

_TODO:_ Draw a `PolyLine` between a launch site to the selected coastline point


In [24]:
# Coordenadas aproximadas para replicar la imagen "Should look like"
launch_site_lat = 28.562302
launch_site_lon = -80.577356
# Punto en la costa bien a la derecha para que la línea cruce el complejo
coastline_lat = 28.56325 
coastline_lon = -80.56795 

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# Marcador de distancia (Exactamente como el de la imagen objetivo)
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12pt; color:#283593;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
    )
)
site_map.add_child(distance_marker)

# Línea azul fina (weight=1 como pide el TODO)
lines = folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]], 
    weight=1,
    color='blue'
)
site_map.add_child(lines)
# 4. Renderizamos para inspección visual
site_map

In [25]:
# 1. RESET: Reiniciamos el mapa para limpiar ejecuciones previas
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# 2. (Opcional) Vuelve a añadir los clústeres si los quieres ver
site_map.add_child(marker_cluster)

# 3. Dibuja SOLO la línea buena
coordinates = [[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]]
lines = folium.PolyLine(locations=coordinates, weight=1, color='blue')
site_map.add_child(lines)

# 4. Añade el marcador de distancia bueno
site_map.add_child(distance_marker)

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


_TODO:_ Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [26]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site
# Coordenadas del Sitio de Lanzamiento (CCAFS LC-40)
launch_site_lat, launch_site_lon = 28.562302, -80.577356

# 1. Autopista más cercana (Samuel C. Phillips Pkwy)
highway_lat, highway_lon = 28.56242, -80.57066

# 2. Vía férrea más cercana (NASA Railroad)
railroad_lat, railroad_lon = 28.57205, -80.58527

# 3. Ciudad más cercana (Titusville)
city_lat, city_lon = 28.61200, -80.80781

In [28]:
# Diccionario con los puntos de interés para iterar
poi_list = [
    {"name": "Highway", "coords": [highway_lat, highway_lon]},
    {"name": "Railway", "coords": [railroad_lat, railroad_lon]},
    {"name": "City", "coords": [city_lat, city_lon]}
]

for poi in poi_list:
    # A. Calcular distancia
    dist = calculate_distance(launch_site_lat, launch_site_lon, poi['coords'][0], poi['coords'][1])
    
    # B. Crear Marcador de Distancia
    distance_marker = folium.Marker(
        poi['coords'],
        icon=DivIcon(
            icon_size=(150,36),
            icon_anchor=(0,0),
            html='<div style="font-size: 12pt; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(dist),
        )
    )
    site_map.add_child(distance_marker)
    
    # C. Dibujar Línea (PolyLine)
    line = folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], poi['coords']],
        weight=1,
        color='blue'
    )
    site_map.add_child(line)
    # Colores para identificar cada tipo de infraestructura
colors = {"Highway": "green", "Railway": "orange", "City": "red", "Coastline": "blue"}

for poi in poi_list:
    dist = calculate_distance(launch_site_lat, launch_site_lon, poi['coords'][0], poi['coords'][1])
    color = colors.get(poi['name'], "black")
    
    # Marcador con NOMBRE y DISTANCIA
    folium.Marker(
        poi['coords'],
        icon=DivIcon(
            icon_size=(200,36),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 11pt; color:{color};"><b>{poi["name"]}: {dist:10.2f} KM</b></div>',
        )
    ).add_child(folium.Popup(poi['name'])).add_to(site_map)
    
    # Línea con el color correspondiente
    folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], poi['coords']],
        weight=2,
        color=color,
        opacity=0.6
    ).add_to(site_map)

site_map

# Mostrar mapa final limpio
site_map

After you plot distance lines to the proximities, you can answer the following questions easily:
- Are launch sites in close proximity to railways?
- Are launch sites in close proximity to highways?
- Are launch sites in close proximity to coastline?
- Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


Joseph Santarcangelo


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2021-05-26|1.0|Yan|Created the initial version|


Copyright © 2021 IBM Corporation. All rights reserved.
